# NFL Salary Cap Efficiency Analysis
Jay Karki | [LinkedIn](https://www.linkedin.com/in/jaykarki/)

Cleaning and merging two datasets into a single flat file covering 320 NFL team-seasons (2013–2022) to analyze how cap allocation relates to on-field performance.

Inputs: `data/raw/NFL Salary By Position Group.xlsx` and `data/raw/team_stats_2003_2023.csv`  
Output: `data/clean/final_nfl_cap_data.csv`

In [ ]:
import pandas as pd
import numpy as np

## Data Ingestion
Loading two datasets: position-group cap data from Spotrac and team performance stats from Pro Football Reference. The cap data came as an Excel file so I converted it to CSV first.

In [ ]:
cap_df = pd.read_excel('../data/raw/NFL Salary By Position Group.xlsx')

cap_df.to_csv('../data/raw/nfl_salary_by_position_group.csv', index=False)

In [ ]:
team_df = pd.read_csv('../data/raw/team_stats_2003_2023.csv')
cap_df = pd.read_csv('../data/raw/nfl_salary_by_position_group.csv')

## Exploratory Data Analysis
Quick look at both raw dataframes before touching anything — shape, columns, dtypes, and null counts.

In [ ]:
print(team_df.head())

print(team_df.columns)

print(team_df.info())

In [ ]:
print(cap_df.head())

print(cap_df.columns)

print(cap_df.info())

## Data Cleaning
Lowercasing all column names and narrowing the cap data down to the 20 columns I need. The two datasets also use different team name formats, which gets handled further down before the merge.

In [ ]:
team_df.columns = team_df.columns.str.lower()

cap_df.columns = cap_df.columns.str.lower()

print(cap_df.columns)

In [ ]:
cap_clean = cap_df[
    [
        'team',
        'season',
        'w',
        'w_pct',
        'playoffs',
        'cap',
        'qb',
        'rb',
        'wr',
        'te',
        'ol',
        'edge',
        'lb',
        'cb',
        'offense',
        'defense',
        'qb_p',
        'wr_p',
        'defense_p',
        'dead_open_specials'
    ]
].copy()

In [ ]:
print(cap_clean.head())

print(cap_clean.info())

## Feature Engineering
Creating three new columns:
- `wins_per_10m_qb` — wins per $10M of QB cap spend, the main efficiency metric for this analysis
- `dead_cap_pct` — dead cap as a percentage of total cap
- `offense_spend_pct` / `defense_spend_pct` — recalculated from raw dollar figures

In [ ]:
cap_clean['wins_per_10m_qb'] = (cap_clean['w'] / (cap_clean['qb'] / 10_000_000)).round(2)
cap_clean['dead_cap_pct'] = (cap_clean['dead_open_specials'] * 100).round(2)
cap_clean['offense_spend_pct'] = cap_clean['offense'] / cap_clean['cap']
cap_clean['defense_spend_pct'] = cap_clean['defense'] / cap_clean['cap']

In [ ]:
print(
    cap_clean[
        [
            'team',
            'season',
            'w',
            'qb',
            'wins_per_10m_qb',
            'dead_cap_pct'
        ]
    ].head()
)

## Team Name Standardization
The cap data uses short names like "Cardinals" while the performance data uses full names like "Arizona Cardinals". Three franchises also relocated during this period — Raiders, Chargers, and Rams — and Washington went through two name changes. The function below maps everything to the correct historical name before the merge.

In [ ]:
team_stats_clean = team_df[
    [
        'year',
        'team',
        'points',
        'points_opp',
        'points_diff',
        'turnover_pct',
        'total_yards',
        'pass_yds',
        'rush_yds'
    ]
].copy().rename(columns={'year': 'season'})

In [ ]:
print(cap_clean['team'].unique()[:10])

print(team_stats_clean['team'].unique()[:10])

In [ ]:
for team in cap_clean['team'].unique():

    matches = [
        other_team
        for other_team in team_stats_clean['team'].unique()
        if team in other_team
    ]

    print(team, '->', matches)

In [ ]:
def fix_team_names(row):

    team = row['team']
    season = row['season']

    if team in ['Raiders', 'Oakland Raiders', 'Las Vegas Raiders']:
        if season <= 2019:
            return 'Oakland Raiders'
        return 'Las Vegas Raiders'

    if team in ['Chargers', 'San Diego Chargers', 'Los Angeles Chargers']:
        if season <= 2016:
            return 'San Diego Chargers'
        return 'Los Angeles Chargers'

    if team in ['Rams', 'St. Louis Rams', 'Los Angeles Rams']:
        if season <= 2015:
            return 'St. Louis Rams'
        return 'Los Angeles Rams'

    if team in ['Commanders', 'Washington Commanders', 'Washington Football Team', 'Washington Redskins']:
        if season <= 2019:
            return 'Washington Redskins'
        if season <= 2021:
            return 'Washington Football Team'
        return 'Washington Commanders'

    team_name_map = {
        'Cardinals': 'Arizona Cardinals',
        'Falcons': 'Atlanta Falcons',
        'Ravens': 'Baltimore Ravens',
        'Bills': 'Buffalo Bills',
        'Panthers': 'Carolina Panthers',
        'Bears': 'Chicago Bears',
        'Bengals': 'Cincinnati Bengals',
        'Browns': 'Cleveland Browns',
        'Cowboys': 'Dallas Cowboys',
        'Broncos': 'Denver Broncos',
        'Lions': 'Detroit Lions',
        'Packers': 'Green Bay Packers',
        'Texans': 'Houston Texans',
        'Colts': 'Indianapolis Colts',
        'Jaguars': 'Jacksonville Jaguars',
        'Chiefs': 'Kansas City Chiefs',
        'Dolphins': 'Miami Dolphins',
        'Vikings': 'Minnesota Vikings',
        'Patriots': 'New England Patriots',
        'Saints': 'New Orleans Saints',
        'Giants': 'New York Giants',
        'Jets': 'New York Jets',
        'Eagles': 'Philadelphia Eagles',
        'Steelers': 'Pittsburgh Steelers',
        '49ers': 'San Francisco 49ers',
        'Seahawks': 'Seattle Seahawks',
        'Buccaneers': 'Tampa Bay Buccaneers',
        'Titans': 'Tennessee Titans'
    }

    return team_name_map.get(team, team)

## Merge
Left join on team and season. Zero nulls across all columns after the merge means every row matched cleanly.

In [ ]:
cap_clean['team'] = cap_clean.apply(fix_team_names, axis=1)

final_df = cap_clean.merge(
    team_stats_clean,
    on=['team', 'season'],
    how='left'
)

print(final_df.isnull().sum())

In [ ]:
print(final_df.head())

print(final_df.info())

print(final_df.describe())

## Export
Rounding floats for cleaner output and saving to `data/clean/`. Final dataset is 320 rows x 31 columns.

In [ ]:
percent_cols = [
    'w_pct',
    'qb_p',
    'wr_p',
    'defense_p',
    'offense_spend_pct',
    'defense_spend_pct',
    'turnover_pct'
]

final_df[percent_cols] = final_df[percent_cols].round(3)
final_df['wins_per_10m_qb'] = final_df['wins_per_10m_qb'].round(2)

In [ ]:
final_df.to_csv(
    '../data/clean/final_nfl_cap_data.csv',
    index=False
)